# 📚 Feature Encoding – Beginner to Intermediate Guide

**What is Encoding?**  
Machine learning models only understand **numbers**, not text.  
Encoding converts text/categorical columns (like `'Male'`, `'Female'`, `'Poor'`, `'Good'`) into numbers.

**Two main types we cover here:**
| Type | When to use | Example |
|------|------------|--------|
| `OrdinalEncoder` | Categories have a natural **order** | Poor < Average < Good |
| `OneHotEncoder` | Categories have **no order** (nominal) | Brand: Maruti, Honda, Ford |

---
## PART 1 – Ordinal Encoding  
Dataset: `customer.csv`  
Columns: `review` (Poor/Average/Good), `education` (School/UG/PG), `purchased` (Yes/No)

In [ ]:
# ── Step 1: Import libraries ──────────────────────────────────────────────────
# numpy  → numerical operations
# pandas → loading and working with tabular data (CSV files)
import numpy as np
import pandas as pd

In [ ]:
# ── Step 2: Load the customer dataset ─────────────────────────────────────────
# This dataset has: age, gender, review, education, purchased
df = pd.read_csv(r'customer.csv')

print("Shape:", df.shape)          # (rows, columns)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ── Step 3: Explore unique values in each column ──────────────────────────────
# Always check what categories exist BEFORE encoding
print("review     :", df['review'].unique())
print("education  :", df['education'].unique())
print("purchased  :", df['purchased'].unique())

In [ ]:
# ── Step 4: Select only the columns we need for encoding ──────────────────────
# We keep: review, education (features) and purchased (target)
# Dropping age and gender for this exercise
df = df[['review', 'education', 'purchased']]
df.head()

In [ ]:
# ── Step 5: Split into Features (X) and Target (y) ───────────────────────────
# X = input columns  (what the model learns FROM)
# y = output column  (what the model tries to PREDICT)
X = df[['review', 'education']]   # features – 2 columns
y = df['purchased']               # target   – 1 column

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# ── Step 6: Train / Test Split ────────────────────────────────────────────────
# WHY split? We train on one part and test on unseen data to check real performance.
# test_size=0.2  → 20% for testing, 80% for training
# random_state=42 → fixes the random shuffle so results are reproducible
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test  size: {X_test.shape[0]}  rows")

In [ ]:
# ── Step 7: OrdinalEncoder – for columns that HAVE a natural order ─────────────
# 
# WHY OrdinalEncoder here?
#   review:    Poor(0) < Average(1) < Good(2)   → order matters!
#   education: School(0) < UG(1) < PG(2)        → order matters!
#
# RULE: fit() on TRAIN only, then transform() BOTH train and test.
#   → fit()    learns the categories from training data
#   → transform() converts text → numbers using what was learned
#   Never fit on test data! That would be DATA LEAKAGE.

from sklearn.preprocessing import OrdinalEncoder

# Define the correct ORDER for each column explicitly
oe = OrdinalEncoder(
    categories=[
        ['Poor', 'Average', 'Good'],   # review:    Poor=0, Average=1, Good=2
        ['School', 'UG', 'PG']        # education: School=0, UG=1, PG=2
    ],
    # If test set has a value not seen in training, raise an error (safe default)
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# Learn categories from training data AND transform it
X_train_enc = oe.fit_transform(X_train)

# ONLY transform test data (DO NOT fit again!)
X_test_enc  = oe.transform(X_test)

print("Learned categories per column:")
for col, cats in zip(['review', 'education'], oe.categories_):
    print(f"  {col}: {list(cats)} → encoded as {list(range(len(cats)))}")

print("\nSample encoded training data (first 5 rows):")
print(pd.DataFrame(X_train_enc, columns=['review_enc', 'education_enc']).head())

In [ ]:
# ── Step 8: LabelEncoder – for the TARGET column (y) ─────────────────────────
#
# LabelEncoder converts a single column of text labels to numbers.
# Used for TARGET (y), not for feature columns (X).
#   'No'  → 0
#   'Yes' → 1

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)   # learn + convert train labels
y_test_enc  = le.transform(y_test)        # only convert test labels

print("Label mapping (index = encoded number):")
for idx, cls in enumerate(le.classes_):
    print(f"  {cls} → {idx}")

print(f"\nOriginal y_train sample: {list(y_train[:5])}")
print(f"Encoded  y_train sample: {y_train_enc[:5]}")

---
## PART 2 – One-Hot Encoding (OHE)  
Dataset: `cars.csv`  
Columns: `brand`, `km_driven`, `fuel`, `owner`, `selling_price`

**Why One-Hot Encoding?**  
Brand names (Maruti, Honda, Ford) have NO natural order.  
If we assign Maruti=1, Honda=2, Ford=3 — the model would WRONGLY think Ford > Honda > Maruti.  
OHE creates a separate **binary (0/1) column** for each category instead.

In [ ]:
# ── Step 9: Load the cars dataset ─────────────────────────────────────────────
df_cars = pd.read_csv(r'cars.csv')

print("Shape:", df_cars.shape)
print("\nColumn info:")
print(df_cars.dtypes)
print("\nFirst 5 rows:")
df_cars.head()

In [ ]:
# ── Step 10: Split Features and Target ────────────────────────────────────────
# selling_price = what we want to predict (target)
# everything else = features the model uses
X = df_cars.drop('selling_price', axis=1)   # drop target from features
y = df_cars['selling_price']                 # target column

print("Feature columns:", X.columns.tolist())
print("Target column  : selling_price")

In [ ]:
# ── Step 11: Train / Test Split ───────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

In [ ]:
# ── Step 12: Brand Bucketing – Group rare brands into 'Other' ─────────────────
#
# PROBLEM: There are 30+ car brands. Most appear very rarely (< 100 times).
# If we OHE every brand, we get 30+ extra columns with mostly 0s (sparse data).
# This hurts model performance (too many useless columns).
#
# SOLUTION: Keep only brands with frequency > 100 in TRAINING data.
# Group all rare brands into a single 'Other' bucket.
#
# IMPORTANT: We compute frequency ONLY on X_train (not X_test)
# to avoid data leakage.

brand_counts    = X_train['brand'].value_counts()        # count each brand in training
frequent_brands = brand_counts[brand_counts > 100].index.tolist()  # brands to keep

print("Frequent brands (freq > 100):")
print(brand_counts[brand_counts > 100])
print(f"\nTotal frequent brands : {len(frequent_brands)}")
print(f"Brands going to 'Other': {brand_counts[brand_counts <= 100].index.tolist()}")

In [ ]:
# ── Step 13: Apply Bucketing to both Train and Test ───────────────────────────
# .copy() prevents the SettingWithCopyWarning – always good practice!
X_train = X_train.copy()
X_test  = X_test.copy()

# If a brand is in our frequent list → keep it as-is
# Otherwise → replace with 'Other'
X_train['brand'] = X_train['brand'].apply(
    lambda x: x if x in frequent_brands else 'Other'
)
X_test['brand'] = X_test['brand'].apply(
    lambda x: x if x in frequent_brands else 'Other'
)

print("Brand distribution in training after bucketing:")
print(X_train['brand'].value_counts())

In [ ]:
# ── Step 14: OneHotEncoder on 'brand' column ──────────────────────────────────
#
# Key parameters explained:
#   drop='first'          → removes the first category to avoid multicollinearity
#                           (if all other dummies are 0, we know it's the first category)
#   sparse_output=False   → return a regular numpy array (not a compressed sparse matrix)
#                           easier to understand and work with
#   handle_unknown='ignore' → if test set has a brand not seen in training, ignore it
#                             (all OHE columns for it become 0)

from sklearn.preprocessing import OneHotEncoder

ohe_brand = OneHotEncoder(
    drop='first',
    sparse_output=False,
    handle_unknown='ignore'
)

# fit_transform on train: learn categories + encode
brand_train = ohe_brand.fit_transform(X_train[['brand']])

# transform only on test: use learned categories to encode
brand_test  = ohe_brand.transform(X_test[['brand']])

# Get the column names created by OHE  (e.g. 'brand_Ford', 'brand_Honda')
brand_cols  = ohe_brand.get_feature_names_out(['brand'])

print(f"Brand columns created: {len(brand_cols)}")
print("Column names:", brand_cols)
print("\nSample brand_train (first 3 rows):")
print(pd.DataFrame(brand_train, columns=brand_cols).head(3))

In [ ]:
# ── Step 15: OneHotEncoder on 'fuel' and 'owner' columns ─────────────────────
#
# Same approach as brand – these are also nominal categories with no order.
# fuel:  Diesel, Petrol, CNG  (no ranking)
# owner: First, Second, Third (could argue ordinal but OHE is safer here)

# --- Fuel ---
ohe_fuel = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
fuel_train = ohe_fuel.fit_transform(X_train[['fuel']])
fuel_test  = ohe_fuel.transform(X_test[['fuel']])
fuel_cols  = ohe_fuel.get_feature_names_out(['fuel'])

# --- Owner ---
ohe_owner = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
owner_train = ohe_owner.fit_transform(X_train[['owner']])
owner_test  = ohe_owner.transform(X_test[['owner']])
owner_cols  = ohe_owner.get_feature_names_out(['owner'])

print(f"fuel  columns : {fuel_cols}")
print(f"owner columns : {owner_cols}")

In [ ]:
# ── Step 16: Combine all encoded columns into one final array ─────────────────
#
# After encoding, we have separate arrays for brand, fuel, owner, and km_driven.
# np.hstack() (horizontal stack) joins them side-by-side into one big array.
#
# Final array column order:
#   [brand OHE columns] + [fuel OHE columns] + [owner OHE columns] + [km_driven]

# Extract the numeric column (km_driven) as a 2D array
km_train = X_train[['km_driven']].values   # .values converts DataFrame → numpy array
km_test  = X_test[['km_driven']].values

# Stack all parts horizontally
X_train_final = np.hstack([brand_train, fuel_train, owner_train, km_train])
X_test_final  = np.hstack([brand_test,  fuel_test,  owner_test,  km_test])

# Build readable column names for the final array
all_cols = list(brand_cols) + list(fuel_cols) + list(owner_cols) + ['km_driven']

print(f"Final X_train_final shape: {X_train_final.shape}")
print(f"Final X_test_final  shape: {X_test_final.shape}")
print(f"Total columns: {len(all_cols)}")
print(f"Column names: {all_cols}")

In [ ]:
# ── Step 17: Preview the final encoded DataFrame ──────────────────────────────
# Convert back to DataFrame just to see it nicely formatted
df_final = pd.DataFrame(X_train_final, columns=all_cols)

print("Final encoded training data (first 5 rows):")
df_final.head()

---
## 📝 Summary – When to Use Which Encoder

| Encoder | Best For | Example |
|---------|----------|---------|
| `OrdinalEncoder` | Ordered categories (features X) | Poor < Average < Good |
| `LabelEncoder` | Target column only (y) | No=0, Yes=1 |
| `OneHotEncoder` | Unordered/nominal categories | Maruti, Honda, Ford |

### ⚠️ Golden Rules
1. **Always fit on TRAINING data only** — never on test data.
2. **Always use transform on test data** — using what was learned from training.
3. **Handle unseen categories** — use `handle_unknown='ignore'` or `'use_encoded_value'`.
4. **Use `.copy()`** when modifying slices of DataFrames to avoid warnings.
5. **OHE with `drop='first'`** to avoid the dummy variable trap (multicollinearity).